
# LangGraph with Databricks LLM - Code Explanation

This code demonstrates a simple conversational AI workflow using **LangGraph** (a graph-based framework for building language model applications) integrated with **Databricks' LLM serving endpoint**.


## 🤖 **Core Function: `call_llm`**

```python
def call_llm(state: MessagesState):
    system_msg = {"role": "system", "content": "Reply only with plain text. No formatting."}
    all_msgs = [system_msg] + state["messages"]
    return {"messages": [llm.invoke(all_msgs)]}
```

**Purpose**: This function processes the conversation state and generates LLM responses.

**How it works**:
1. **System Message**: Adds a system prompt instructing the LLM to respond in plain text only (no markdown/formatting)
2. **Message Preparation**: Combines the system message with existing conversation messages
3. **LLM Invocation**: Calls the Databricks LLM with the complete message history
4. **State Update**: Returns the new message to be added to the conversation state

## 🕸️ **Graph Construction**

```python
builder = StateGraph(MessagesState)  # Create graph with message state management
builder.add_node("call_llm", call_llm)  # Add the LLM processing node
builder.add_edge(START, "call_llm")  # Connect start to LLM node
builder.add_edge("call_llm", END)  # Connect LLM node to end
graph = builder.compile()  # Compile the graph for execution
```

**Graph Flow**: `START → call_llm → END`

This creates a simple linear workflow where:
- The conversation starts
- Messages are processed by the LLM
- The conversation ends

## 🚀 **Execution**

```python
messages = graph.invoke({"messages": [HumanMessage("Tell me more about Databricks ?")]})
```

**What happens**:
1. **Input**: Creates a human message asking about Databricks
2. **Processing**: The graph processes this through the `call_llm` node
3. **Output**: Returns the complete conversation including the LLM's response
4. **Result**: The `messages` variable contains both the input question and the generated answer

## 🎯 **Key Concepts**

- **StateGraph**: Manages conversation state and message flow
- **MessagesState**: Built-in state type for handling conversation messages
- **Node**: A processing unit in the graph (here, the LLM call)
- **Edges**: Define the flow between nodes
- **Invoke**: Executes the graph with given input

## 💡 **Use Cases**

This pattern is useful for:
- Building conversational AI applications
- Creating chatbots with Databricks LLMs
- Implementing structured conversation flows
- Integrating with larger AI workflows

The code represents a foundational building block that can be extended with additional nodes for more complex behaviors like tool calling, memory management, or multi-step reasoning.

In [0]:
from dotenv import load_dotenv
import os
import random
from typing import Literal, TypedDict
from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage
from langgraph.graph.message import add_messages
from typing_extensions import Annotated
from databricks_langchain import ChatDatabricks
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph import MessagesState


llm = ChatDatabricks(endpoint = "databricks-gpt-oss-120b")


# In LangGraph, MessageState is a built-in helper class (or type) that represents the state of messages flowing through a LangGraph state graph — especially useful when you’re building conversational or agentic workflows where messages (like chat history) are passed between nodes.

def call_llm(state: MessagesState):

    system_msg = {"role": "system", "content": "Reply only with plain text. No formatting."}
    all_msgs = [system_msg] + state["messages"]

    return {"messages": [llm.invoke(all_msgs)]}


builder = StateGraph(MessagesState)
builder.add_node("call_llm", call_llm)


builder.add_edge(START, "call_llm")
builder.add_edge("call_llm", END)

graph = builder.compile()

messages = graph.invoke({"messages": [HumanMessage("Tell me more about Databricks ?")]})

In [0]:
message = messages['messages'][-1]

for part in message.content:
    if part.get("type") == "text":
       ai_message = part.get("text", "")

print(ai_message)

Databricks is a cloud‑based data‑engineering and analytics platform that was founded in 2013 by the creators of Apache Spark. The company builds on top of Spark, providing a managed service that lets teams develop, run, and scale data pipelines, machine‑learning models, and analytics workloads without having to maintain their own Spark clusters.

Key components of the Databricks platform include:

- **Unified Data Analytics Workspace** – an interactive environment with collaborative notebooks (supporting Python, Scala, SQL, R, and Java) that lets data engineers, analysts, and data scientists work together in real time.
- **Databricks Runtime** – a performance‑tuned Spark engine that includes optimizations, built‑in libraries, and automatic scaling. It also offers specialized runtimes for machine learning and genomics.
- **Delta Lake** – an open‑source storage layer that brings ACID transactions, schema enforcement, and time‑travel capabilities to data lakes on cloud storage (AWS S3, Azure Data Lake Storage, Google Cloud Storage). This makes lakehouse architectures possible, combining the reliability of a data warehouse with the flexibility of a data lake.
- **MLflow** – an open‑source framework for the full machine‑learning lifecycle, handling experiment tracking, model packaging, and deployment. Databricks hosts a managed MLflow service that integrates with the rest of the platform.
- **Data Engineering and ETL** – tools such as Auto Loader for incremental ingestion, Structured Streaming for real‑time processing, and Delta Live Tables for declarative pipeline development and monitoring.
- **SQL Analytics / Lakehouse** – a SQL‑native interface that lets analysts run ad‑hoc queries, build dashboards, and serve BI tools directly against the lakehouse data.
- **Security and Governance** – fine‑grained role‑based access control, Unity Catalog for centralized data governance, and integration with enterprise identity providers.

Databricks runs on the major public clouds—AWS, Microsoft Azure, and Google Cloud. On Azure it is offered as “Azure Databricks,” tightly integrated with Azure services such as Azure Data Lake Storage, Azure Synapse, and Azure Active Directory. On AWS it integrates with S3, IAM, and other AWS services.

Typical use cases include:

1. Large‑scale ETL and data‑pipeline automation, where teams need to ingest, transform, and store petabytes of data.
2. Advanced analytics and machine‑learning model development, leveraging Spark’s distributed compute and the built‑in MLflow capabilities.
3. Real‑time streaming analytics for IoT, clickstream, or fraud‑detection scenarios using Structured Streaming.
4. Business intelligence and reporting, where analysts query curated Delta Lake tables via SQL and connect BI tools like Tableau, Power BI, or Looker.
5. Data‑science collaboration, with shared notebooks, version control, and reproducible experiments.

Pricing is consumption‑based. Customers pay for compute (Databricks Units or DBUs) and storage, with options for on‑demand auto‑scaling clusters, job clusters, and serverless compute. There are also tiered plans (Standard, Premium, and Enterprise) that add features such as Unity Catalog, higher security compliance, and dedicated support.

Because the platform abstracts away much of the operational complexity of Spark, many organizations adopt Databricks to accelerate time‑to‑value on big‑data projects, unify disparate data workloads under a single “lakehouse” architecture, and enable cross‑functional collaboration between engineers, analysts, and data scientists.